In [1]:
pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 88.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
Quantum Capsule Network — Full Research Validation v4 (QCN-Adaptive)
=====================================================================
CHANGES vs original:
  - All figure backgrounds → WHITE
  - Fig 1: only gradient variance plot, NO horizontal/vertical reference lines
  - Fig 4: Final Expressibility (polar) removed
           S-score + Depth Evolution combined into ONE dual-axis graph
  - Loss convergence: runs until loss < LOSS_CONVERGE_THR or EPOCHS_QFIM_MAX
  - MAX_DEPTH = 10  (adaptive layers grow up to 8-10)

IMAGE SUPPORT (NEW):
  - 64×64 input image → PCA দিয়ে n = n_caps * cap_size dimension এ reduce
  - load_and_reduce_images() function যোগ করা হয়েছে
  - run_qfim_training() এ X_input, y_input parameter যোগ করা হয়েছে
  - USE_CUSTOM_IMAGES = True করলে নিজের image ব্যবহার করা যাবে

PLOT CHANGES (v4.3):
  - Fig1: title removed, fit lines removed, vertical error bars removed
  - Fig2: right panel removed, left panel title removed
  - Fig3: split into 2 separate figures, titles removed
  - Fig4 left: title removed, X=Depth Y=S-score scatter
  - Fig4 right: title removed, threshold line removed, label cleaned
"""

import numpy as _np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings, os, csv, time as _time, datetime
warnings.filterwarnings('ignore')

import pennylane as qml
from pennylane import numpy as pnp
from scipy.linalg import eigvalsh
from scipy.stats import ks_2samp
from scipy.optimize import curve_fit
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA          # ← NEW: PCA import

_np.random.seed(42)
pnp.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
# Output directory + CSV helpers
# ════════════════════════════════════════════════════════════════════════════
BASE_DIR = r"/kaggle/working/"
os.makedirs(BASE_DIR, exist_ok=True)

_TS      = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
_TIMINGS = {}

def _csv_path(name):
    return os.path.join(BASE_DIR, f"{name}_{_TS}.csv")

def _write_csv(path, header, rows):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    kb = os.path.getsize(path) // 1024
    print(f"  ✓  CSV → {os.path.basename(path)}  ({kb} KB)")

def _tick(label):
    _TIMINGS[label] = _time.perf_counter()

def _tock(label):
    elapsed = _time.perf_counter() - _TIMINGS[label]
    _TIMINGS[label] = elapsed
    print(f"  ⏱  [{label}] finished in {elapsed:.1f}s  ({elapsed/60:.2f} min)")

# ════════════════════════════════════════════════════════════════════════════
# Style — WHITE background
# ════════════════════════════════════════════════════════════════════════════
BG      = 'white'
AX_BG   = 'white'
AX_EDGE = '#cccccc'
TICK_C  = '#444444'
TXT_C   = '#111111'
GRID_C  = '#e8e8e8'

plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    AX_BG,
    'axes.edgecolor':    AX_EDGE,
    'axes.labelcolor':   TXT_C,
    'xtick.color':       TICK_C,
    'ytick.color':       TICK_C,
    'grid.color':        GRID_C,
    'grid.linewidth':    0.8,
    'text.color':        TXT_C,
    'legend.facecolor':  'white',
    'legend.edgecolor':  AX_EDGE,
    'legend.fontsize':   9,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'font.family':       'monospace',
    'lines.linewidth':   2.0,
})

# ════════════════════════════════════════════════════════════════════════════
# Hyperparameters
# ════════════════════════════════════════════════════════════════════════════
QUBIT_RANGES   = [4, 6, 8, 10, 12, 14, 16, 18 ,20]
CAP_SIZES      = [2, 3, 4,5, 6, 9]
CAP_COLORS     = {2: '#1a7abf', 3: '#e07b00', 4: '#1e8c45', 6: '#9b2dca', 9: '#c0392b'}
C_GLOBAL       = '#2c2c2c'
CAP_PAL        = ['#1a7abf', '#c0392b', '#e07b00', '#1e8c45']

FIXED_N        = 12
N_SEEDS        = 5
N_LAYERS       = 2
N_STEPS        = 4
DEPTH_MAX      = 6
N_ENT_SEEDS    = 5
N_HESS_SEEDS   = 3
K_HESS         = 4

MAX_DEPTH      = 10

QFIM_INTERVAL  = 5
S_PLATEAU_THR  = 0.30
QFIM_N         = 12
QFIM_N_CAPS    = 4
QFIM_CAP_SIZE  = 3

EPOCHS_QFIM_MAX    = 300
LOSS_CONVERGE_THR  = 0.15

ADAPT_WARMUP_EPOCHS = 5

# ════════════════════════════════════════════════════════════════════════════
# IMAGE SUPPORT CONFIG  (NEW)
# ════════════════════════════════════════════════════════════════════════════
# নিজের 64×64 image ব্যবহার করতে চাইলে:
#   1. USE_CUSTOM_IMAGES = True করুন
#   2. CUSTOM_IMAGE_PATH এ আপনার image folder path দিন
#      (অথবা নিচে load_custom_images() function customize করুন)
# False রাখলে আগের মতো sklearn digits dataset ব্যবহার হবে
USE_CUSTOM_IMAGES  = False
CUSTOM_IMAGE_PATH  = r"C:\Users\User\Downloads\model validation result"   # ← আপনার path

# ════════════════════════════════════════════════════════════════════════════
# [NEW] Image Loading + Dimension Reduction Functions
# ════════════════════════════════════════════════════════════════════════════

def load_and_reduce_images(image_array, labels, n_caps, cap_size,
                           n_classes=2, test_size=0.3, random_state=42):
    """
    64×64 (বা যেকোনো size) image কে PCA দিয়ে circuit-ready format এ আনে।

    Parameters
    ----------
    image_array  : np.ndarray, shape (N, H, W) বা (N, H*W)
                   যেকোনো size এর grayscale image
    labels       : np.ndarray, shape (N,)  — binary label (0 বা 1)
    n_caps       : int — number of capsules
    cap_size     : int — qubits per capsule
    n_classes    : int — কতটা class রাখবে (binary হলে 2)
    test_size    : float — train/test split ratio
    random_state : int

    Returns
    -------
    X_train : np.ndarray, shape (N_train, n_qubits)  [0, π] range এ scaled
    X_test  : np.ndarray, shape (N_test,  n_qubits)
    y_train : np.ndarray, shape (N_train,) — {-1, +1} labels
    y_test  : np.ndarray, shape (N_test,)
    pca     : fitted PCA object (later use এর জন্য)
    """
    n_qubits = n_caps * cap_size

    # ── Step 1: Flatten ───────────────────────────────────────────────────
    if image_array.ndim == 3:
        # (N, H, W) → (N, H*W)
        X_flat = image_array.reshape(len(image_array), -1).astype(float)
    else:
        X_flat = image_array.astype(float)

    print(f"  [ImageReduce] Raw shape   : {image_array.shape}")
    print(f"  [ImageReduce] Flat shape  : {X_flat.shape}")

    # ── Step 2: Binary class filter ───────────────────────────────────────
    if n_classes == 2:
        unique_labels = _np.unique(labels)
        if len(unique_labels) > 2:
            # প্রথম দুটো class নিই
            mask = (labels == unique_labels[0]) | (labels == unique_labels[1])
            X_flat = X_flat[mask]
            labels = labels[mask]
            # 0/1 এ convert করি
            labels = (labels == unique_labels[1]).astype(int)
            print(f"  [ImageReduce] Binary filter: class {unique_labels[0]} vs {unique_labels[1]}")

    # ── Step 3: Normalize ─────────────────────────────────────────────────
    X_norm = normalize(X_flat, norm='l2')

    # ── Step 4: Train/Test split ──────────────────────────────────────────
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_norm, labels, test_size=test_size,
        random_state=random_state, stratify=labels)

    # ── Step 5: PCA — n_qubits dimension এ reduce ─────────────────────────
    n_components = min(n_qubits, X_tr.shape[0], X_tr.shape[1])
    if n_components < n_qubits:
        print(f"  [ImageReduce] WARNING: n_components capped at {n_components} "
              f"(requested {n_qubits})")

    pca = PCA(n_components=n_components, random_state=random_state)
    X_tr_pca = pca.fit_transform(X_tr)   # fit only on train!
    X_te_pca = pca.transform(X_te)

    var_explained = pca.explained_variance_ratio_.sum()
    print(f"  [ImageReduce] PCA target  : {n_qubits} components")
    print(f"  [ImageReduce] Variance ✓  : {var_explained:.4f}  "
          f"({var_explained*100:.1f}%)")
    print(f"  [ImageReduce] Reduced shape: {X_tr_pca.shape}")

    # ── Step 6: [0, π] range এ scale করি ─────────────────────────────────
    # Per-feature min/max from train set → apply to test set
    X_min = X_tr_pca.min(axis=0)
    X_max = X_tr_pca.max(axis=0)
    denom = X_max - X_min + 1e-8

    X_tr_scaled = (X_tr_pca - X_min) / denom * _np.pi
    X_te_scaled = _np.clip((X_te_pca - X_min) / denom * _np.pi, 0, _np.pi)

    # ── Step 7: Label → {-1, +1} ─────────────────────────────────────────
    y_train_pm = (2 * y_tr - 1).astype(float)
    y_test_pm  = (2 * y_te - 1).astype(float)

    print(f"  [ImageReduce] Train: {X_tr_scaled.shape}, "
          f"Test: {X_te_scaled.shape}")
    print(f"  [ImageReduce] Label dist train: "
          f"+1={(_np.array(y_train_pm)==1).sum()}  "
          f"-1={(_np.array(y_train_pm)==-1).sum()}")

    return X_tr_scaled, X_te_scaled, y_train_pm, y_test_pm, pca


def load_custom_images(image_path, n_caps, cap_size):
    """
    নিজের image folder থেকে load করার জন্য।
    image_path এর ভেতরে দুটো subfolder থাকবে: class0/ এবং class1/

    আপনার data structure অনুযায়ী এই function customize করুন।

    Returns
    -------
    images : np.ndarray (N, 64, 64)
    labels : np.ndarray (N,)
    """
    try:
        import cv2  # যদি opencv থাকে
        has_cv2 = True
    except ImportError:
        has_cv2 = False

    images_list = []
    labels_list = []

    for label_idx, class_name in enumerate(['class0', 'class1']):
        class_dir = os.path.join(image_path, class_name)
        if not os.path.exists(class_dir):
            print(f"  WARNING: {class_dir} not found, skipping")
            continue

        for fname in os.listdir(class_dir):
            if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                continue
            fpath = os.path.join(class_dir, fname)
            if has_cv2:
                img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (64, 64))
            else:
                from PIL import Image
                img = Image.open(fpath).convert('L').resize((64, 64))
                img = _np.array(img)

            images_list.append(img.astype(float) / 255.0)
            labels_list.append(label_idx)

    images = _np.array(images_list)
    labels = _np.array(labels_list)
    print(f"  Loaded {len(images)} images from {image_path}")
    return images, labels




print('=' * 70)
print('  QUANTUM CAPSULE NETWORK — FULL RESEARCH VALIDATION v4.3 (IMAGE SUPPORT)')
print('=' * 70)
print(f'  Qubits       : {QUBIT_RANGES}')
print(f'  Cap sizes    : {CAP_SIZES}')
print(f'  Initial depth: {N_LAYERS}  Max depth: {MAX_DEPTH}')
print(f'  GV seeds     : {N_SEEDS}   Entropy seeds: {N_ENT_SEEDS}')
print(f'  Hessian      : {N_HESS_SEEDS} seeds')
print(f'  QFIM epochs  : up to {EPOCHS_QFIM_MAX}  (stop when loss < {LOSS_CONVERGE_THR})')
print(f'  Custom images: {USE_CUSTOM_IMAGES}')
print(f'  Output dir   : {BASE_DIR}')
print('=' * 70)


# ════════════════════════════════════════════════════════════════════════════
# Circuit builders
# ════════════════════════════════════════════════════════════════════════════

def build_global_circuit(n, dev, depth=N_LAYERS):
    n_p = depth * n * 2

    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        x_vals = [float(x[i]) for i in range(n)]
        p = params.reshape(depth, n, 2)
        for i in range(n):
            qml.RY(x_vals[i], wires=i)
        for layer in range(depth):
            for i in range(n):
                qml.RY(p[layer, i, 0], wires=i)
            for i in range(n - 1):
                qml.CNOT(wires=[i, i + 1])
            qml.CNOT(wires=[0, n - 1])
            for i in range(n):
                qml.RZ(p[layer, i, 1], wires=i)
        return qml.expval(qml.PauliZ(0) @ qml.PauliZ(n - 1))

    return circuit, n_p


def build_adaptive_qcn_circuit(n, n_caps, cap_size, depths, dev):
    offsets = [0]
    for c in range(n_caps):
        offsets.append(offsets[-1] + depths[c] * cap_size * 4)
    total_params = offsets[-1]
    max_d = max(depths)

    @qml.qnode(dev, diff_method='best')
    def circuit(params, x):
        for i in range(n):
            qml.Hadamard(wires=i)

        for layer in range(max_d):
            for c in range(n_caps):
                if layer >= depths[c]:
                    continue
                w  = list(range(c * cap_size, (c + 1) * cap_size))
                cp = params[offsets[c]: offsets[c + 1]].reshape(
                         depths[c], cap_size, 4)
                xc = [float(x[c * cap_size + j]) for j in range(cap_size)]

                for j, wire in enumerate(w):
                    qml.RY(xc[j] * cp[layer, j, 3], wires=wire)
                for j, wire in enumerate(w):
                    qml.Rot(cp[layer, j, 0],
                            cp[layer, j, 1],
                            cp[layer, j, 2], wires=wire)
                if layer % 2 == 0:
                    for j in range(len(w) - 1):
                        qml.CNOT(wires=[w[j], w[j + 1]])
                    if len(w) > 2:
                        qml.CNOT(wires=[w[-1], w[0]])

            if layer % 2 == 0:
                ic_offset = (layer // 2) % 2
                for c in range(ic_offset, n_caps - 1, 2):
                    if layer < depths[c] and layer < depths[c + 1]:
                        boundary_a = (c + 1) * cap_size - 1
                        boundary_b = (c + 1) * cap_size
                        qml.CNOT(wires=[boundary_a, boundary_b])

        return qml.math.stack([
            qml.expval(qml.PauliZ(c * cap_size)) for c in range(n_caps)
        ])

    return circuit, total_params, offsets


# ════════════════════════════════════════════════════════════════════════════
# AdaptiveQCN
# ════════════════════════════════════════════════════════════════════════════

def _ppc(depth, cap_size):
    return depth * cap_size * 4


def _init_cap_params(depth, cap_size, rng=None):
    if rng is None:
        rng = _np.random
    p       = rng.uniform(0, 0.5, _ppc(depth, cap_size))
    p[3::4] = rng.uniform(0.8, 1.2, len(p) // 4)
    return p.astype(float)


def _compute_qfim_score(cap_params_np, depth, x_cap_np, cap_size):
    dev       = qml.device('default.qubit', wires=cap_size)
    flat_size = _ppc(depth, cap_size)
    x_arr_np  = _np.array(x_cap_np, dtype=float)

    @qml.qnode(dev)
    def cap_state(p_flat):
        p = p_flat.reshape(depth, cap_size, 4)
        for i in range(cap_size):
            qml.Hadamard(wires=i)
        for layer in range(depth):
            for i in range(cap_size):
                qml.RY(float(x_arr_np[i]) * p[layer, i, 3], wires=i)
            for i in range(cap_size):
                qml.Rot(p[layer, i, 0], p[layer, i, 1], p[layer, i, 2], wires=i)
            if layer % 2 == 0:
                for i in range(cap_size - 1):
                    qml.CNOT(wires=[i, i + 1])
                if cap_size > 2:
                    qml.CNOT(wires=[cap_size - 1, 0])
        return qml.state()

    try:
        mt_fn  = qml.metric_tensor(cap_state, approx='block-diag')
        p_flat = pnp.array(cap_params_np.flatten(), requires_grad=True)
        qfim   = mt_fn(p_flat).reshape(flat_size, flat_size)
        eigs   = _np.clip(_np.abs(_np.linalg.eigvalsh(_np.array(qfim))), 1e-8, 1e4)
        d_eff  = float((_np.sum(eigs) ** 2) / (_np.sum(eigs ** 2) + 1e-12))
        kappa  = float(_np.max(eigs) / (_np.min(eigs) + 1e-8))
        s      = d_eff / (_np.log10(kappa + 1) + 1e-10)
        return float(s), float(kappa), float(d_eff)
    except Exception:
        return 0.0, 1.0, 0.0


class AdaptiveQCN:
    def __init__(self, n, n_caps, cap_size, dev,
                 init_depth=N_LAYERS, max_depth=MAX_DEPTH,
                 plateau_thr=S_PLATEAU_THR, seed=0):
        self.n           = n
        self.n_caps      = n_caps
        self.cap_size    = cap_size
        self.dev         = dev
        self.max_depth   = max_depth
        self.plateau_thr = plateau_thr

        rng              = _np.random.RandomState(seed)
        self.depths      = [init_depth] * n_caps
        self._cap_params = [_init_cap_params(init_depth, cap_size, rng)
                            for _ in range(n_caps)]
        self._s_window   = [[] for _ in range(n_caps)]
        self.s_records   = []
        self.growth_log  = []
        self._rebuild_params()

    def _rebuild_params(self):
        flat          = _np.concatenate(self._cap_params)
        self.params   = pnp.array(flat, requires_grad=True)
        _, self.n_params, self.offsets = build_adaptive_qcn_circuit(
            self.n, self.n_caps, self.cap_size, self.depths, self.dev)

    def _sync_cap_params_from_flat(self, flat_np):
        for c in range(self.n_caps):
            sz = _ppc(self.depths[c], self.cap_size)
            self._cap_params[c] = flat_np[
                self.offsets[c]: self.offsets[c] + sz].copy()

    def get_circuit(self):
        return build_adaptive_qcn_circuit(
            self.n, self.n_caps, self.cap_size, self.depths, self.dev)

    def update_params(self, new_flat):
        new_np = _np.array(new_flat)
        self._sync_cap_params_from_flat(new_np)
        self.params = pnp.array(new_np, requires_grad=True)

    def maybe_grow(self, x_sample_np, epoch, verbose=False):
        grew = False
        for c in range(self.n_caps):
            x_cap = x_sample_np[c * self.cap_size: (c + 1) * self.cap_size]
            s, _, _ = _compute_qfim_score(
                self._cap_params[c], self.depths[c], x_cap, self.cap_size)
            self.s_records.append((epoch, c, s))
            self._s_window[c].append(s)

            if self.depths[c] >= self.max_depth:
                continue
            sw = self._s_window[c]
            if len(sw) >= 4:
                dS = (sw[-1] - sw[-4]) / 4
                if abs(dS) < self.plateau_thr:
                    old_d = self.depths[c]
                    self.depths[c] += 1
                    rng     = _np.random.RandomState(epoch * 100 + c)
                    new_lyr = rng.uniform(0, 0.1, self.cap_size * 4)
                    new_lyr[3::4] = rng.uniform(0.9, 1.1, self.cap_size)
                    self._cap_params[c] = _np.concatenate(
                        [self._cap_params[c], new_lyr])
                    self.growth_log.append((epoch, c, old_d, self.depths[c]))
                    grew = True
                    if verbose:
                        print(f'    [AdaptiveQCN] Ep {epoch:3d} | '
                              f'Cap{c} {old_d}→{self.depths[c]} '
                              f'(|ΔS|={abs(dS):.3f})')
        if grew:
            self._rebuild_params()
        return grew

    def final_s_scores(self, x_sample_np):
        scores = []
        for c in range(self.n_caps):
            x_cap = x_sample_np[c * self.cap_size: (c + 1) * self.cap_size]
            s, _, _ = _compute_qfim_score(
                self._cap_params[c], self.depths[c], x_cap, self.cap_size)
            scores.append(s)
        return scores


# ════════════════════════════════════════════════════════════════════════════
# Helper utilities
# ════════════════════════════════════════════════════════════════════════════

def exp_decay(n, A, alpha):
    return A * _np.exp(-alpha * _np.array(n, dtype=float))


def safe_fit(xs, ys, label):
    try:
        popt, _ = curve_fit(exp_decay, xs, ys, p0=[0.5, 0.3], maxfev=10000,
                            bounds=([0, 1e-4], [1e3, 10]))
        print(f'    Fit {label:<18}: A={popt[0]:.5f}  α={popt[1]:.5f}')
        return popt
    except Exception as e:
        print(f'    Fit {label:<18}: FAILED — {e}')
        return None


def savefig(name):
    path = os.path.join(BASE_DIR, name)
    plt.savefig(path, dpi=200, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close()
    kb = os.path.getsize(path) // 1024
    print(f'  ✓  Figure → {name}  ({kb} KB)')


# ════════════════════════════════════════════════════════════════════════════
# [1] Gradient Variance Scaling
# ════════════════════════════════════════════════════════════════════════════

def grad_var_single(n, n_caps, cap_size, model, seed=0):
    pnp.random.seed(seed)
    _np.random.seed(seed)
    dev = qml.device('lightning.qubit', wires=n)

    if model == 'global':
        circ, n_p = build_global_circuit(n, dev)

        def cost(p, x):
            return circ(p, x) ** 2

        p   = pnp.random.uniform(0, 2 * _np.pi, n_p, requires_grad=True)
        x   = pnp.array(_np.random.uniform(0, _np.pi, n), requires_grad=False)
        gvs = []
        for _ in range(N_STEPS):
            g = qml.grad(cost, argnums=0)(p, x)
            gvs.append(float(_np.var(_np.array(g).flatten())))
        raw  = float(_np.mean(gvs))
        norm = raw / n_p
        return raw, norm, n_p

    else:
        rng   = _np.random.RandomState(seed)
        x_np  = rng.uniform(0, _np.pi, n)
        x_pnp = pnp.array(x_np, requires_grad=False)

        aqcn  = AdaptiveQCN(n, n_caps, cap_size, dev,
                            init_depth=N_LAYERS, seed=seed)

        ep_per_layer = max(1, ADAPT_WARMUP_EPOCHS // N_LAYERS)
        params_full  = _np.array(aqcn.params).copy()
        act_per_cap  = cap_size * 4

        for d_stage in range(N_LAYERS):
            n_act = (d_stage + 1) * act_per_cap
            n_frz = (N_LAYERS - d_stage - 1) * act_per_cap

            active_parts = []
            frozen_parts = []
            for c in range(n_caps):
                base = aqcn.offsets[c]
                active_parts.append(params_full[base: base + n_act])
                if n_frz > 0:
                    frozen_parts.append(params_full[base + n_act:
                                                     base + n_act + n_frz])

            p_active = pnp.array(
                _np.concatenate(active_parts), requires_grad=True)
            p_frozen = pnp.array(
                _np.concatenate(frozen_parts) if frozen_parts else _np.array([]),
                requires_grad=False)

            opt   = qml.AdamOptimizer(stepsize=0.015)
            n_ep  = (ep_per_layer if d_stage < N_LAYERS - 1
                     else ADAPT_WARMUP_EPOCHS - d_stage * ep_per_layer)
            n_ep  = max(1, n_ep)
            circ, _, _ = aqcn.get_circuit()

            for ep in range(n_ep):
                def cost_warmup(pa, c=circ, xx=x_pnp,
                                pf=p_frozen,
                                nc=n_caps, na=n_act, nf=n_frz):
                    chunks = []
                    for ci in range(nc):
                        chunks.append(pa[ci * na: (ci + 1) * na])
                        if nf > 0:
                            chunks.append(pf[ci * nf: (ci + 1) * nf])
                    pp = pnp.concatenate(chunks)
                    return pnp.sum(c(pp, xx) ** 2)

                p_active, _ = opt.step_and_cost(cost_warmup, p_active)

            p_act_np = _np.array(p_active)
            p_frz_np = _np.array(p_frozen)
            for c in range(n_caps):
                base = aqcn.offsets[c]
                params_full[base: base + n_act] = p_act_np[c * n_act:
                                                             (c+1) * n_act]
                if n_frz > 0:
                    params_full[base + n_act: base + n_act + n_frz] = \
                        p_frz_np[c * n_frz: (c+1) * n_frz]

            aqcn.update_params(params_full)
            ep_global = (d_stage + 1) * ep_per_layer
            if ep_global % QFIM_INTERVAL == 0:
                aqcn.maybe_grow(x_np, ep_global, verbose=False)

        circ_fn, _, _ = aqcn.get_circuit()

        def cost_gv(p, xx=x_pnp, c=circ_fn):
            return pnp.sum(c(p, xx) ** 2)

        gvs = []
        for _ in range(N_STEPS):
            g = qml.grad(cost_gv, argnums=0)(aqcn.params)
            gvs.append(float(_np.var(_np.array(g).flatten())))
        raw  = float(_np.mean(gvs))
        norm = raw / aqcn.n_params
        return raw, norm, aqcn.n_params


# ── Run Section 1 ─────────────────────────────────────────────────────────────
print('\n[1/5] Gradient Variance Scaling ...')
_tick('1_GradVar')

gv_global = {'n': [], 'mean': [], 'std': [], 'norm': []}
gv_caps   = {cs: {'n': [], 'mean': [], 'std': [], 'norm': []}
             for cs in CAP_SIZES}

for n in QUBIT_RANGES:
    print(f'  n={n:<3}', end='  |  ')
    raws = [grad_var_single(n, 1, n, 'global', seed=s)[0] for s in range(N_SEEDS)]
    gv_global['n'].append(n)
    gv_global['mean'].append(_np.mean(raws))
    gv_global['std'].append(_np.std(raws))
    gv_global['norm'].append(_np.mean(raws) / n)
    print(f'global={_np.mean(raws):.2e}', end='  ')

    for cs in CAP_SIZES:
        if n % cs != 0 or (n % cs == 0 and n // cs == 1):
            continue
        nc = n // cs
        rs, ns_ = [], []
        for s in range(N_SEEDS):
            r, nm, _ = grad_var_single(n, nc, cs, 'qcn', seed=s)
            rs.append(r)
            ns_.append(nm)
        gv_caps[cs]['n'].append(n)
        gv_caps[cs]['mean'].append(_np.mean(rs))
        gv_caps[cs]['std'].append(_np.std(rs))
        gv_caps[cs]['norm'].append(_np.mean(ns_))
        print(f'cs={cs}:{_np.mean(rs):.2e}', end='  ')
    print()

print()
decay_fits      = {}
decay_fits_norm = {}
decay_fits['global']      = safe_fit(gv_global['n'], gv_global['mean'],  'global(raw)')
decay_fits_norm['global'] = safe_fit(gv_global['n'], gv_global['norm'],  'global(norm)')
for cs in CAP_SIZES:
    d = gv_caps[cs]
    if len(d['n']) < 3:
        decay_fits[cs] = decay_fits_norm[cs] = None
        print(f'    Fit cs={cs}(raw)       : skipped (< 3 points)')
        continue
    decay_fits[cs]      = safe_fit(d['n'], d['mean'], f'cs={cs}(raw)')
    decay_fits_norm[cs] = safe_fit(d['n'], d['norm'], f'cs={cs}(norm)')

_tock('1_GradVar')

# ── Save Section 1 CSV ────────────────────────────────────────────────────────
_rows_gv = []
for i, n in enumerate(gv_global['n']):
    a_raw  = decay_fits['global'][1]      if decay_fits.get('global')      is not None else ''
    a_norm = decay_fits_norm['global'][1] if decay_fits_norm.get('global') is not None else ''
    _rows_gv.append([n, 'global',
                     gv_global['mean'][i], gv_global['std'][i],
                     gv_global['norm'][i], a_raw, a_norm])

for cs in CAP_SIZES:
    d = gv_caps[cs]
    for i, n in enumerate(d['n']):
        a_raw  = decay_fits[cs][1]      if decay_fits.get(cs)      is not None else ''
        a_norm = decay_fits_norm[cs][1] if decay_fits_norm.get(cs) is not None else ''
        _rows_gv.append([n, f'qcn_cs{cs}',
                         d['mean'][i], d['std'][i], d['norm'][i],
                         a_raw, a_norm])

_write_csv(_csv_path('01_gradient_variance'),
           ['n_qubits', 'model', 'grad_var_mean', 'grad_var_std',
            'grad_var_norm', 'alpha_exp_fit_raw', 'alpha_exp_fit_norm'],
           _rows_gv)


# ════════════════════════════════════════════════════════════════════════════
# [2] Entanglement Entropy vs Depth
# ════════════════════════════════════════════════════════════════════════════

def entropy_from_state(rho):
    evs = _np.real(_np.linalg.eigvalsh(_np.array(rho)))
    evs = _np.abs(evs)
    evs = evs[evs > 1e-12]
    evs /= evs.sum()
    return float(-_np.sum(evs * _np.log2(evs + 1e-15)))


def entropy_global_at_depth(n, depth, n_seeds=15):
    dev   = qml.device('default.qubit', wires=n)
    half  = n // 2
    sub_w = list(range(half))
    results = []
    for seed in range(n_seeds):
        rng = _np.random.RandomState(seed)
        p   = rng.uniform(0.3, 2.5, depth * n * 2)

        @qml.qnode(dev)
        def circ(params, _d=depth, _n=n):
            for i in range(_n):
                qml.Hadamard(wires=i)
            for d in range(_d):
                for i in range(_n):
                    qml.RY(params[d * _n * 2 + i],      wires=i)
                    qml.RZ(params[d * _n * 2 + _n + i], wires=i)
                for i in range(_n - 1):
                    qml.CNOT(wires=[i, i + 1])
                qml.CNOT(wires=[0, _n - 1])
            return qml.density_matrix(wires=sub_w)

        results.append(entropy_from_state(circ(_np.array(p))))
    return float(_np.mean(results)), float(_np.std(results))


def entropy_adaptive_qcn_at_depth(n, n_caps, cap_size, depth, n_seeds=15):
    """
    Random (Haar-like) parameter sampling দিয়ে entropy মাপা।
    """
    half  = n // 2
    sub_w = list(range(half))
    results = []

    for seed in range(n_seeds):
        rng    = _np.random.RandomState(seed)
        x_np   = rng.uniform(0, _np.pi, n)

        offs = [0]
        for c in range(n_caps):
            offs.append(offs[-1] + depth * cap_size * 4)
        total_params = offs[-1]

        params_np = rng.uniform(0, 2 * _np.pi, total_params)
        params_np[3::4] = rng.uniform(0.8, 1.2, len(params_np[3::4]))

        dev_dm = qml.device('default.qubit', wires=n)

        @qml.qnode(dev_dm)
        def circ_dm(params, _depth=depth, _offs=offs):
            for i in range(n):
                qml.Hadamard(wires=i)
            for layer in range(_depth):
                for c in range(n_caps):
                    w  = list(range(c * cap_size, (c + 1) * cap_size))
                    cp = params[_offs[c]: _offs[c+1]].reshape(_depth, cap_size, 4)
                    xc = [float(x_np[c * cap_size + j]) for j in range(cap_size)]
                    for j, wire in enumerate(w):
                        qml.RY(xc[j] * cp[layer, j, 3], wires=wire)
                    for j, wire in enumerate(w):
                        qml.Rot(cp[layer, j, 0],
                                cp[layer, j, 1],
                                cp[layer, j, 2], wires=wire)
                    if layer % 2 == 0:
                        for j in range(len(w) - 1):
                            qml.CNOT(wires=[w[j], w[j + 1]])
                        if len(w) > 2:
                            qml.CNOT(wires=[w[-1], w[0]])
                if layer % 2 == 0:
                    ic_offset = (layer // 2) % 2
                    for c in range(ic_offset, n_caps - 1, 2):
                        if layer < depth:
                            qml.CNOT(wires=[(c + 1) * cap_size - 1,
                                            (c + 1) * cap_size])
            return qml.density_matrix(wires=sub_w)

        results.append(entropy_from_state(circ_dm(params_np)))

    return float(_np.mean(results)), float(_np.std(results))


# ── Run Section 2 ─────────────────────────────────────────────────────────────
print('\n[2/5] Entanglement Entropy (n=12, depth-proxy 1-6) ...')
_tick('2_Entropy')

depths_e = list(range(1, DEPTH_MAX + 1))
S_global = {'mean': [], 'std': []}
S_caps   = {cs: {'mean': [], 'std': []}
            for cs in CAP_SIZES if FIXED_N % cs == 0}
valid_cs = list(S_caps.keys())

for d in depths_e:
    m, s = entropy_global_at_depth(FIXED_N, d, N_ENT_SEEDS)
    S_global['mean'].append(m)
    S_global['std'].append(s)
    for cs in S_caps:
        nc   = FIXED_N // cs
        m, s = entropy_adaptive_qcn_at_depth(FIXED_N, nc, cs, d, N_ENT_SEEDS)
        S_caps[cs]['mean'].append(m)
        S_caps[cs]['std'].append(s)
    print(f'  d≈{d}  global={S_global["mean"][-1]:.3f}  '
          + '  '.join(f'cs={cs}:{S_caps[cs]["mean"][-1]:.3f}'
                      for cs in S_caps))

_tock('2_Entropy')

# ── Save Section 2 CSV ────────────────────────────────────────────────────────
_rows_ent = []
for di, d in enumerate(depths_e):
    _rows_ent.append([d, 'global', FIXED_N, 1,
                      S_global['mean'][di], S_global['std'][di]])
    for cs in valid_cs:
        _rows_ent.append([d, f'qcn_cs{cs}', FIXED_N, FIXED_N // cs,
                          S_caps[cs]['mean'][di], S_caps[cs]['std'][di]])

_write_csv(_csv_path('02_entanglement_entropy'),
           ['depth_proxy', 'model', 'n_qubits', 'n_caps',
            'entropy_mean_bits', 'entropy_std_bits'],
           _rows_ent)


# ════════════════════════════════════════════════════════════════════════════
# [3] Hessian KS-Statistic Scaling
# ════════════════════════════════════════════════════════════════════════════

def hessian_eigs_adaptive(n, n_caps, cap_size, n_seeds=6):
    eigs_all = []
    for seed in range(n_seeds):
        _np.random.seed(seed * 100 + n)
        pnp.random.seed(seed * 100 + n)
        rng   = _np.random.RandomState(seed * 100 + n)
        x_np  = rng.uniform(0, _np.pi, n)
        x_pnp = pnp.array(x_np, requires_grad=False)

        dev  = qml.device('lightning.qubit', wires=n)
        aqcn = AdaptiveQCN(n, n_caps, cap_size, dev,
                           init_depth=N_LAYERS, seed=seed)
        opt  = qml.AdamOptimizer(stepsize=0.015)

        for ep in range(ADAPT_WARMUP_EPOCHS):
            circ_fn, _, _ = aqcn.get_circuit()

            def cost_h(pp, c=circ_fn, xx=x_pnp):
                return pnp.sum(c(pp, xx) ** 2)

            new_p, _ = opt.step_and_cost(cost_h, aqcn.params)
            aqcn.update_params(_np.array(new_p))
            if ep % QFIM_INTERVAL == 0:
                grew = aqcn.maybe_grow(x_np, ep, verbose=False)
                if grew:
                    opt = qml.AdamOptimizer(stepsize=0.015)

        k       = min(aqcn.n_params, K_HESS)
        eps     = 1e-4
        circ_fn, _, _ = aqcn.get_circuit()

        def cost_fn(pp, c=circ_fn, xx=x_pnp):
            return pnp.sum(c(pp, xx) ** 2)

        H = _np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                pp_arr = _np.array(aqcn.params)
                pp_arr[i] += eps; pp_arr[j] += eps
                pp = pnp.array(pp_arr); v_pp = float(cost_fn(pp))
                pp_arr[j] -= 2 * eps
                pp = pnp.array(pp_arr); v_pm = float(cost_fn(pp))
                pp_arr[i] -= 2 * eps
                pp = pnp.array(pp_arr); v_mm = float(cost_fn(pp))
                pp_arr[j] += 2 * eps
                pp = pnp.array(pp_arr); v_mp = float(cost_fn(pp))
                H[i, j] = (v_pp - v_pm - v_mp + v_mm) / (4 * eps ** 2)
        eigs_all.extend(_np.abs(eigvalsh(H)).tolist())
    return _np.array(eigs_all)


def hessian_eigs_global(n, n_seeds=6):
    eigs_all = []
    dev = qml.device('lightning.qubit', wires=n)
    k   = min(n, K_HESS)
    eps = 1e-4
    for seed in range(n_seeds):
        pnp.random.seed(seed * 100 + n)
        x    = pnp.array(_np.random.uniform(0, _np.pi, n), requires_grad=False)
        circ, n_p = build_global_circuit(n, dev)

        def cost(p, x=x):
            return circ(p, x) ** 2

        p = pnp.random.uniform(0, 2 * _np.pi, n_p, requires_grad=True)
        H = _np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                pp = p.copy(); pp[i] += eps; pp[j] += eps
                pm = p.copy(); pm[i] += eps; pm[j] -= eps
                mp = p.copy(); mp[i] -= eps; mp[j] += eps
                mm = p.copy(); mm[i] -= eps; mm[j] -= eps
                H[i, j] = (cost(pp) - cost(pm) - cost(mp) + cost(mm)) / (4 * eps ** 2)
        eigs_all.extend(_np.abs(eigvalsh(H)).tolist())
    return _np.array(eigs_all)


# ── Run Section 3 ─────────────────────────────────────────────────────────────
print('\n[3/5] Hessian KS-Statistic Scaling ...')
_tick('3_HessianKS')

ks_table = {cs: {'n': [], 'stat': [], 'pval': []} for cs in CAP_SIZES}
for n in QUBIT_RANGES:
    print(f'  n={n:<3}', end='  |  ')
    eigs_g = hessian_eigs_global(n, N_HESS_SEEDS)
    for cs in CAP_SIZES:
        if n % cs != 0:
            continue
        nc         = n // cs
        eigs_c     = hessian_eigs_adaptive(n, nc, cs, N_HESS_SEEDS)
        stat, pval = ks_2samp(eigs_g, eigs_c)
        ks_table[cs]['n'].append(n)
        ks_table[cs]['stat'].append(stat)
        ks_table[cs]['pval'].append(pval)
        print(f'cs={cs} D={stat:.3f}(p={pval:.3f})', end='  ')
    print()

_tock('3_HessianKS')

# ── Save Section 3 CSV ────────────────────────────────────────────────────────
_rows_ks = []
for cs in CAP_SIZES:
    d = ks_table[cs]
    for i, n in enumerate(d['n']):
        pval = d['pval'][i]
        sig  = ('***' if pval < 0.001 else '**' if pval < 0.01
                else '*' if pval < 0.05 else 'ns')
        _rows_ks.append([n, f'qcn_cs{cs}', d['stat'][i], pval, sig])

_write_csv(_csv_path('03_hessian_ks_stat'),
           ['n_qubits', 'model', 'ks_stat_D', 'ks_pval', 'significance'],
           _rows_ks)


# ════════════════════════════════════════════════════════════════════════════
# [4] QFIM Adaptive Depth vs Matched-Depth Baseline
#     UPDATED: 64×64 image support via PCA
# ════════════════════════════════════════════════════════════════════════════

def run_qfim_training(n_caps, cap_size, adaptive=True,
                      fixed_depth=2, label='adaptive',
                      X_input=None, y_input=None):
    """
    Parameters
    ----------
    n_caps, cap_size : circuit config
    adaptive         : True = AdaptiveQCN, False = fixed depth baseline
    fixed_depth      : depth for baseline (ignored if adaptive=True)
    label            : print label
    X_input          : np.ndarray (N, H, W) বা (N, H*W) — custom images
                       None হলে sklearn digits fallback
    y_input          : np.ndarray (N,) — binary labels (0/1)
                       None হলে sklearn digits fallback
    """
    n = n_caps * cap_size

    # ── Data Loading ───────────────────────────────────────────────────────
    if X_input is not None and y_input is not None:
        # ══ Custom image path ══
        print(f'  [{label}] Using custom images → PCA reduce to {n} features')
        X_train, X_test, y_train, y_test, _pca = load_and_reduce_images(
            X_input, y_input, n_caps, cap_size)
        # training এ শুধু X_train ব্যবহার করব
    else:
        # ══ Fallback: sklearn digits ══
        print(f'  [{label}] Using sklearn digits dataset')
        X, y            = load_digits(n_class=2, return_X_y=True)
        X               = normalize(X, norm='l2')
        Xtr, _, ytr, _  = train_test_split(X, y, test_size=0.3, random_state=42)
        X_train         = Xtr[:, :n] * _np.pi
        y_train         = (2 * ytr - 1).astype(float)

    x_sample = X_train[0]

    # ── Circuit + Optimizer setup ──────────────────────────────────────────
    dev = qml.device('lightning.qubit', wires=n)
    vn  = pnp.array(_np.ones(n_caps) / n_caps)
    opt = qml.AdamOptimizer(stepsize=0.015)

    if adaptive:
        aqcn = AdaptiveQCN(n, n_caps, cap_size, dev,
                           init_depth=1, max_depth=MAX_DEPTH,
                           plateau_thr=S_PLATEAU_THR, seed=0)
    else:
        aqcn = AdaptiveQCN(n, n_caps, cap_size, dev,
                           init_depth=fixed_depth, max_depth=fixed_depth,
                           plateau_thr=S_PLATEAU_THR, seed=0)

    depth_history = []
    loss_log      = []

    for epoch in range(EPOCHS_QFIM_MAX):
        circ_fn, _, _ = aqcn.get_circuit()

        def cost_fn(p):
            # Batch size 64 — কম noise
            batch_size = min(64, len(X_train))
            idx   = _np.random.choice(len(X_train), batch_size, replace=False)
            total = pnp.array(0.0)
            for xi, yt in zip(X_train[idx], y_train[idx]):
                xi_pnp = pnp.array(xi, requires_grad=False)
                pred   = pnp.sum(vn * circ_fn(p, xi_pnp))
                total  = total + (pred - yt) ** 2
            return total / batch_size

        new_p, loss_val = opt.step_and_cost(cost_fn, aqcn.params)
        aqcn.update_params(_np.array(new_p))
        loss_log.append(float(loss_val))
        depth_history.append(list(aqcn.depths))

        if epoch % QFIM_INTERVAL == 0 and adaptive:
            grew = aqcn.maybe_grow(x_sample, epoch, verbose=True)
            if grew:
                opt = qml.AdamOptimizer(stepsize=0.015)

        if (epoch + 1) % 20 == 0:
            print(f'    [{label}] Ep {epoch+1:3d} | '
                  f'depths={aqcn.depths} | loss={float(loss_val):.4f}')

        # ── Early stop when converged ──────────────────────────────────────
        if float(loss_val) < LOSS_CONVERGE_THR:
            print(f'    [{label}] Converged at epoch {epoch+1} '
                  f'(loss={float(loss_val):.4f} < {LOSS_CONVERGE_THR})')
            break

    final_s = aqcn.final_s_scores(x_sample)
    return {
        's_records':     aqcn.s_records,
        'depth_history': _np.array(depth_history),
        'growth_log':    aqcn.growth_log,
        'final_depths':  list(aqcn.depths),
        'loss_log':      loss_log,
        'final_s':       final_s,
        'epochs_run':    len(loss_log),
    }


# ── Run Section 4 ─────────────────────────────────────────────────────────────
print(f'\n[4/5] QFIM Adaptive Depth vs Matched-Depth Baseline '
      f'(n={QFIM_N}, {QFIM_N_CAPS}×{QFIM_CAP_SIZE}) ...')
_tick('4_QFIM')

# ══════════════════════════════════════════════════════════════
#  IMAGE LOADING
#  USE_CUSTOM_IMAGES = True হলে নিজের 64×64 image load হবে
#  False হলে sklearn digits fallback
# ══════════════════════════════════════════════════════════════
if USE_CUSTOM_IMAGES:
    print('  Loading custom 64×64 images ...')
    _custom_images, _custom_labels = load_custom_images(
        CUSTOM_IMAGE_PATH, QFIM_N_CAPS, QFIM_CAP_SIZE)
    X_for_qfim = _custom_images
    y_for_qfim = _custom_labels
else:
    print('  Using sklearn digits dataset (USE_CUSTOM_IMAGES=False)')
    X_for_qfim = None
    y_for_qfim = None

print('  Running adaptive ...')
qfim_adaptive = run_qfim_training(
    QFIM_N_CAPS, QFIM_CAP_SIZE,
    adaptive=True,
    label='adaptive',
    X_input=X_for_qfim,
    y_input=y_for_qfim)

mean_final_depth = int(round(_np.mean(qfim_adaptive['final_depths'])))
print(f'  Adaptive final depths: {qfim_adaptive["final_depths"]}  '
      f'→ baseline depth = {mean_final_depth}')
print(f'  Running fixed-depth-{mean_final_depth} baseline ...')

qfim_fixed = run_qfim_training(
    QFIM_N_CAPS, QFIM_CAP_SIZE,
    adaptive=False,
    fixed_depth=mean_final_depth,
    label=f'fixed-{mean_final_depth}',
    X_input=X_for_qfim,
    y_input=y_for_qfim)

_tock('4_QFIM')

EPOCHS_A = qfim_adaptive['epochs_run']
EPOCHS_F = qfim_fixed['epochs_run']

# ── Save Section 4 CSVs ───────────────────────────────────────────────────────
_rows_loss = []
for ep, la in enumerate(qfim_adaptive['loss_log']):
    _rows_loss.append([ep, 'adaptive', la])
for ep, lf in enumerate(qfim_fixed['loss_log']):
    _rows_loss.append([ep, f'fixed_{mean_final_depth}', lf])

_write_csv(_csv_path('04a_qfim_loss_log'),
           ['epoch', 'model', 'loss'], _rows_loss)

_cap_cols = [f'cap{c}_depth' for c in range(QFIM_N_CAPS)]
_rows_depth = []
dh_a = qfim_adaptive['depth_history']
dh_f = qfim_fixed['depth_history']
for ep in range(EPOCHS_A):
    _rows_depth.append(['adaptive', ep] + list(dh_a[ep]))
for ep in range(EPOCHS_F):
    _rows_depth.append([f'fixed_{mean_final_depth}', ep] + list(dh_f[ep]))

_write_csv(_csv_path('04b_qfim_depth_history'),
           ['model', 'epoch'] + _cap_cols, _rows_depth)

_rows_s = []
for (ep, cap, s) in qfim_adaptive['s_records']:
    _rows_s.append(['adaptive', ep, cap, s])
for (ep, cap, s) in qfim_fixed['s_records']:
    _rows_s.append([f'fixed_{mean_final_depth}', ep, cap, s])

_write_csv(_csv_path('04c_qfim_s_score_records'),
           ['model', 'epoch', 'capsule', 's_score'], _rows_s)

_rows_grow = []
for (ep, cap, old_d, new_d) in qfim_adaptive['growth_log']:
    _rows_grow.append([ep, cap, old_d, new_d])
_write_csv(_csv_path('04d_qfim_growth_log'),
           ['epoch', 'capsule', 'depth_before', 'depth_after'], _rows_grow)

_rows_fs = []
for c, s in enumerate(qfim_adaptive['final_s']):
    _rows_fs.append([c, qfim_adaptive['final_depths'][c], s])
_write_csv(_csv_path('04e_qfim_final_s_scores'),
           ['capsule', 'final_depth', 'final_s_score'], _rows_fs)



# ════════════════════════════════════════════════════════════════════════════
# Results summary
# ════════════════════════════════════════════════════════════════════════════
print('\n' + '═' * 72)
print('  RESULTS SUMMARY  (v4.3 — image support)')
print('═' * 72)

fl_a  = qfim_adaptive['loss_log'][-1]
fl_f  = qfim_fixed['loss_log'][-1]
gain  = (fl_f - fl_a) / (abs(fl_f) + 1e-9) * 100

print(f'\n[1] Gradient Variance Fits')
g_alpha = decay_fits['global'][1] if decay_fits.get('global') is not None else None
if g_alpha:
    print(f'    Global  α={g_alpha:.4f}')
for cs in CAP_SIZES:
    if decay_fits.get(cs) is not None:
        a_r  = decay_fits[cs][1]
        slow = g_alpha / a_r if g_alpha else float('nan')
        print(f'    QCN cs={cs}  α={a_r:.4f}  slowdown={slow:.1f}×')

print(f'\n[2] Entropy (n={FIXED_N}, d={DEPTH_MAX})')
g_S = S_global['mean'][-1]
print(f'    Global : {g_S:.3f} bits')
for cs in valid_cs:
    c_S = S_caps[cs]['mean'][-1]
    print(f'    cs={cs}  : {c_S:.3f} bits  ratio={g_S/c_S:.2f}×')

print(f'\n[3] Hessian KS')
for cs in CAP_SIZES:
    d = ks_table[cs]
    if d['n']:
        sig = ('***' if d["pval"][-1] < 0.001 else '**' if d["pval"][-1] < 0.01
               else '*' if d["pval"][-1] < 0.05 else 'ns')
        print(f'    cs={cs}  D={d["stat"][-1]:.4f}  p={d["pval"][-1]:.4f}  {sig}')

print(f'\n[4] QFIM')
print(f'    Custom images used     : {USE_CUSTOM_IMAGES}')
print(f'    Adaptive final depths  : {qfim_adaptive["final_depths"]}')
print(f'    Adaptive epochs run    : {EPOCHS_A}')
print(f'    Fixed epochs run       : {EPOCHS_F}')
print(f'    Final loss adaptive    : {fl_a:.4f}')
print(f'    Final loss baseline    : {fl_f:.4f}')
print(f'    Adaptive improvement   : {gain:+.1f}%')


# ════════════════════════════════════════════════════════════════════════════
# Timing + summary CSV
# ════════════════════════════════════════════════════════════════════════════
_TIMINGS['TOTAL'] = sum(v for v in _TIMINGS.values())

print('\n' + '═' * 72)
print('  TIMING SUMMARY')
print('═' * 72)
for sec, secs in _TIMINGS.items():
    print(f'  {sec:<22} {secs:>10.1f}s  ({secs/60:.2f} min)')

_rows_timing = [[k, round(v, 2), round(v/60, 3)] for k, v in _TIMINGS.items()]
_write_csv(_csv_path('pennylane_timing'),
           ['section', 'wall_seconds', 'wall_minutes'], _rows_timing)

_rows_sum = [['experiment', 'metric', 'value', 'unit']]
if decay_fits.get('global') is not None:
    _rows_sum.append(['GradVar', 'global_alpha_raw', decay_fits['global'][1], 'exp_decay_rate'])
for cs in CAP_SIZES:
    if decay_fits.get(cs) is not None:
        g_a = decay_fits['global'][1] if decay_fits.get('global') else 1.0
        _rows_sum.append(['GradVar', f'qcn_cs{cs}_alpha_raw', decay_fits[cs][1], 'exp_decay_rate'])
        _rows_sum.append(['GradVar', f'qcn_cs{cs}_slowdown', g_a / decay_fits[cs][1], 'ratio'])
_rows_sum.append(['Entropy', 'global_final', S_global['mean'][-1], 'bits'])
for cs in valid_cs:
    _rows_sum.append(['Entropy', f'qcn_cs{cs}_final', S_caps[cs]['mean'][-1], 'bits'])
for cs in CAP_SIZES:
    d = ks_table[cs]
    if d['n']:
        _rows_sum.append(['HessKS', f'qcn_cs{cs}_D', d['stat'][-1], 'KS_stat'])
_rows_sum += [
    ['QFIM', 'custom_images',       USE_CUSTOM_IMAGES,                'bool'],
    ['QFIM', 'adaptive_final_loss', fl_a,                             'loss'],
    ['QFIM', 'fixed_final_loss',    fl_f,                             'loss'],
    ['QFIM', 'improvement_pct',     gain,                             'percent'],
    ['QFIM', 'growth_events',       len(qfim_adaptive['growth_log']), 'count'],
]
_write_csv(_csv_path('00_summary'),
           ['experiment', 'metric', 'value', 'unit'], _rows_sum)

print('\n' + '═' * 72)
print('  OUTPUT FILES →', BASE_DIR)
print('═' * 72)
for fname in sorted(os.listdir(BASE_DIR)):
    fpath = os.path.join(BASE_DIR, fname)
    kb    = os.path.getsize(fpath) // 1024
    print(f'  {fname:<60} {kb:>5} KB')
print('═' * 72)
print('\n  v4.3-image-support complete. ✓')

  QUANTUM CAPSULE NETWORK — FULL RESEARCH VALIDATION v4.3 (IMAGE SUPPORT)
  Qubits       : [4, 6, 8, 10, 12, 14, 16, 18, 20]
  Cap sizes    : [2, 3, 4, 5, 6, 9]
  Initial depth: 2  Max depth: 10
  GV seeds     : 5   Entropy seeds: 5
  Hessian      : 3 seeds
  QFIM epochs  : up to 300  (stop when loss < 0.15)
  Custom images: False
  Output dir   : /kaggle/working/

[1/5] Gradient Variance Scaling ...
  n=4    |  global=3.18e-03  cs=2:1.21e-03  
  n=6    |  global=2.63e-05  cs=2:7.79e-03  cs=3:4.89e-03  
  n=8    |  global=5.38e-06  cs=2:6.46e-03  cs=4:1.67e-03  
  n=10   |  global=6.57e-05  cs=2:9.24e-03  cs=5:1.84e-03  
  n=12   |  global=7.14e-08  cs=2:9.01e-03  cs=3:5.96e-03  cs=4:1.60e-03  cs=6:1.93e-04  
  n=14   |  global=3.63e-09  cs=2:1.35e-02  
  n=16   |  global=7.13e-13  cs=2:1.18e-02  cs=4:1.36e-03  
  n=18   |  global=2.59e-15  cs=2:1.14e-02  cs=3:6.69e-03  cs=6:4.24e-04  cs=9:7.67e-04  
  n=20   |  global=1.53e-14  cs=2:1.09e-02  cs=4:3.13e-03  cs=5:1.11e-03  

    Fit gl

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()